# 1. SETUP & ENVIRONMENT INITIALIZATION


In [18]:
from google.colab import drive
import os
import torch
import shutil
from pathlib import Path
import pandas as pd

In [19]:
# Clone project repository from GitHub
!git clone https://github.com/rhejna/docile-anomaly-extraction.git

# Navigate into the project root directory
%cd docile-anomaly-extraction

Cloning into 'docile-anomaly-extraction'...
remote: Enumerating objects: 179, done.
remote: Counting objects: 100% (179/179), done.
remote: Compressing objects: 100% (109/109), done.
remote: Total 179 (delta 94), reused 144 (delta 66), pack-reused 0 (from 0)
Receiving objects: 100% (179/179), 50.22 KiB | 12.55 MiB/s, done.
Resolving deltas: 100% (94/94), done.
/content/docile-anomaly-extraction/docile-anomaly-extraction/docile-anomaly-extraction/docile-anomaly-extraction


Uncomment and run this only if the `docile-anomaly-extraction` folder is nested, as a result of the repository having been cloned multiple times ⬇

In [32]:
# Navigate to the global root directory /content
# %cd /content

# # If nested folders exist, clear everything to start afresh
# if Path("docile-anomaly-extraction").exists():
#     # Cleanly clone a brand-new repository into a clean temporary folder
#     !rm -rf clean_repo
#     !git clone https://github.com/rhejna/docile-anomaly-extraction.git clean_repo
#     !rm -rf docile-anomaly-extraction
#     !mv clean_repo docile-anomaly-extraction

# # Move permanently to the correct location
# %cd docile-anomaly-extraction
# print("Dossier de travail actuel nettoyé :", Path.cwd())

/content
Cloning into 'clean_repo'...
remote: Enumerating objects: 184, done.
remote: Counting objects: 100% (184/184), done.
remote: Compressing objects: 100% (115/115), done.
remote: Total 184 (delta 98), reused 144 (delta 65), pack-reused 0 (from 0)
Receiving objects: 100% (184/184), 50.81 KiB | 10.16 MiB/s, done.
Resolving deltas: 100% (98/98), done.
/content/docile-anomaly-extraction
Dossier de travail actuel nettoyé : /content/docile-anomaly-extraction


In [20]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 2. DATA PREPARATION (LOCAL SSD CACHING)

In [33]:
# Create the local directory structure if it does not exist
Path("data/raw/sroie").mkdir(parents=True, exist_ok=True)

# Copy from Google Drive to the local Colab drive
print("Copying SROIE data from Google Drive to the local drive...")
!cp -r "/content/drive/MyDrive/docile-anomaly-extraction/data/raw/sroie/." "data/raw/sroie/"
print("Copy completed successfully!")

# Quick check of the contents
print("Contents of data/raw/sroie:", os.listdir("data/raw/sroie"))

# Define checkpoint path pointing directly to Google Drive
CHECKPOINT_DIR = "/content/drive/MyDrive/docile-anomaly-extraction/data/checkpoints/best_model"

Copying SROIE data from Google Drive to the local drive...
Copy completed successfully!
Contents of data/raw/sroie: ['task3test', 'task2train', 'text.task1_2test', 'task1train', 'task1_2test']


# 3. HARDWARE & DEPENDENCY CHECK

In [22]:
# Check if CUDA (GPU) is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if device.type == "cpu":
    print("WARNING: GPU is not enabled! Please go to Runtime -> Change runtime type and select a T4 GPU.")
else:
    print(f"GPU detected: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU detected: Tesla T4


In [39]:
# Install required libraries while preserving Colab's built-in PyTorch distribution
!pip install -r requirements.txt

  Using cached certifi-2026.7.22-py3-none-any.whl.metadata (2.5 kB)
  Using cached colorama-0.4.6-py2.py3-none-any.whl.metadata (17 kB)
  Using cached filelock-3.32.3-py3-none-any.whl.metadata (2.0 kB)
  Using cached fsspec-2026.7.0-py3-none-any.whl.metadata (10 kB)
  Using cached hf_xet-1.6.0-cp38-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (4.9 kB)
  Using cached huggingface_hub-1.32.0-py3-none-any.whl.metadata (16 kB)
  Using cached iniconfig-2.3.0-py3-none-any.whl.metadata (2.5 kB)
  Using cached markupsafe-3.0.3-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.7 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached pillow-12.3.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (9.1 kB)
  Using cached pluggy-1.6.0-py3-none-any.whl.metadata (4.8 kB)
  Using cached pytesseract-0.3.13-py3-none-any.whl.metadata (11 kB)
  Using cached pytest-9.1.1-py3-none-any.whl.metadata (7.6

# 4. TRAINING PIPELINE EXECUTION

In [34]:
# Run the training script across all dataset samples (omit --limit to use all 563 training receipts)
# If you encounter 'CUDA out of memory', reduce batch_size to 2 in run_training.py
!python -m src.model.run_training --limit 563

--- 1. Loading data ---
Dataset limited to 563 receipts.
Train samples: 506 | Val samples: 57
--- 2. Initializing processor and datasets ---
--- 3. Setting up model, optimizer, and device ---
Using device: cuda
Loading weights: 100% 212/212 [00:00<00:00, 4038.66it/s]
[transformers] LayoutLMv3ForTokenClassification LOAD REPORT from: microsoft/layoutlmv3-base
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
--- 4. Starting training loop ---
Writing model shards: 100% 1/1 [00:01<00:00,  1.55s/it]
Writing model shards: 100% 1/1 [00:05<00:00,  5.35s/it]
--- 5. Saving training history ---
Training history successfully saved to data/results/training_history.json


# 5. EVALUATION & MARKDOWN TABLE GENERATION

In [42]:
!apt-get update && apt-get install -y tesseract-ocr
!pip install pytesseract

In [43]:
# Run baseline and LayoutLMv3 evaluations
!python -m src.run_evaluation
!python -m src.model.evaluate --checkpoint /content/drive/MyDrive/docile-anomaly-extraction/data/checkpoints/best_model

Detailed results saved to: data/results/baseline_regex_2026-11-04.json
Scores log updated: data/results/scores_log.csv
Loading checkpoint from /content/drive/MyDrive/docile-anomaly-extraction/data/checkpoints/best_model...
Loading weights: 100% 214/214 [00:00<00:00, 2354.39it/s]
Running evaluation on all 63 validation receipts...

--- Evaluation Test Scores Comparison ---
Field      | Raw (%)    | Cleaned (%)  | Gap (%) 
--------------------------------------------------
company    |      66.70 |        66.70 |    +0.00
date       |      25.40 |        79.40 |   +54.00
address    |      66.70 |        66.70 |    +0.00
total      |      81.00 |        84.10 |    +3.10
Detailed results saved to: data/results/layoutlmv3_cleaned_2026-11-04.json
Scores log updated: data/results/scores_log.csv


In [44]:
# Load the scores from the CSV log and generate the comparison table
from src.eval.compare import build_comparison_table

log_path = Path("data/results/scores_log.csv")
if log_path.exists():
    df = pd.read_csv(log_path)

    # Retrieve the baseline if it exists
    base_subset = df[df["method"] == "baseline_regex"]
    baseline_scores = base_subset.iloc[-1][["company", "date", "address", "total", "global"]].to_dict() if not base_subset.empty else {}

    # Retrieve LayoutLMv3 if it exists
    model_subset = df[df["method"].str.contains("layoutlmv3", na=False)]
    model_scores = model_subset.iloc[-1][["company", "date", "address", "total", "global"]].to_dict() if not model_subset.empty else {}

    if baseline_scores and model_scores:
        print("\n=== MARKDOWN COMPARISON TABLE FOR README ===")
        print(build_comparison_table(baseline_scores, model_scores))
    else:
        print("\nNotice: Missing records in scores_log.csv to build the full comparison table.")
else:
    print(f"\nError: {log_path} was not found. Check if evaluation scripts ran successfully.")


=== MARKDOWN COMPARISON TABLE FOR README ===
| Field | Baseline (%) | LayoutLMv3 (%) | Gap (%) |
| :--- | :---: | :---: | :---: |
| Company | 33.50% | 66.70% | +33.20% |
| Date | 60.50% | 79.40% | +18.90% |
| Address | 0.20% | 66.70% | +66.50% |
| Total | 18.50% | 84.10% | +65.60% |
| Global | 28.20% | 74.20% | +46.00% |


# 6. ARTIFACT BACKUP TO GOOGLE DRIVE

In [45]:
# 1. Copy the checkpoints to Google Drive
drive_checkpoint_dir = Path("/content/drive/MyDrive/docile-anomaly-extraction/data/checkpoints")
drive_checkpoint_dir.mkdir(parents=True, exist_ok=True)

print("Copying checkpoints to Google Drive...")
!cp -r "data/checkpoints/." "{drive_checkpoint_dir}/"

# 2. Copy the results history to Google Drive
drive_results_dir = Path("/content/drive/MyDrive/docile-anomaly-extraction/data/results")
drive_results_dir.mkdir(parents=True, exist_ok=True)

print("Copying training history to Google Drive...")
!cp -r "data/results/." "{drive_results_dir}/"

print("All artifacts successfully safely backed up to Google Drive!")

Copying checkpoints to Google Drive...
Copying training history to Google Drive...
All artifacts successfully safely backed up to Google Drive!
